## STEP 0 
Import Libraries, Connect to Snowflake, & Initialize NLP Pipelines

In [ ]:
## TODO TODO
## 1. Make a robust, immune-to-timeout solution for querying
## 2. Figure out if you need some kind of `while true... except StopIteration` pattern like Gemini said
## 3. Finish the 5-24 to 5-31 chunk of NER analysis you still need to do
## 4. Get the merge-components of the sentiment analysis working
## 5. The functions to iterate through source data and writeback in batches is not DRY enough-- fix that



from datasets import Dataset
from datetime import datetime, timedelta
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import torch
from transformers import pipeline

# this basically means "smoke em if you got em" where the "em" is NVIDIA GPU
DEVICE = 0 if torch.cuda.is_available() else -1

SF_USR = os.getenv('SF_USR')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')

# connect to database and init a cursor for querying
xct_params = {
    "user":                 SF_USR
   ,"account":              SF_ID
   ,"warehouse":            SF_WH
   ,"database":             SF_DB
   ,"schema":               SF_SC
   ,"role":                 SF_RL
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}

def connect_to_snowflake(connection_parameters: dict) -> (snowflake.connector.connection.SnowflakeConnection
                                                         ,snowflake.connector.cursor.SnowflakeCursor):
    SF_XCT = snowflake.connector.connect(**xct_params)
    return SF_XCT.cursor()

CSR = connect_to_snowflake(xct_params)

def execute_query(conn: snowflake.connector.connection.SnowflakeConnection
                 ,cursor: snowflake.connector.cursor.SnowflakeCursor
                 ,query: str):
    try:
        cursor.execute(query)
    except ForbiddenError as e:
        print(f"Caught Forbidden Error: {e}")
        print(f"Attempting to re-establish Snowflake connection...")
    
# sentiment analyzer doo-dad instantiation
PIPL_SNT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=DEVICE,
    truncation=True,
    max_length = 512 
)
## This sentiment pipeline returns labels like ['LABEL_0', 'LABEL_1', 'LABEL_2']
## instead of ['Negative', 'Neutral', 'Positive']
## The below-linked mapping indicates which model-labels match to which
## human-understandable terms. 
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=256 
)

Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


## STEP 1
Ingest Post-Text into Memory

In [8]:
query = f"""
select content_id
      ,usa_timestamp as post_created_usa_timestamp
      ,post_text
from {SF_DB}.{SF_SC}.firehose_processed
where (first_detected_language = 'English'
   or  first_detected_language is null
   )
  and post_created_usa_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                     and to_timestamp_tz('2025-05-31 23:59:59+0000')
;
"""
# and post_created_usa_timestamp >= (select nvl(max(post_created_usa_timestamp), '1900-00-00 00:00:00+0000)
#                                    from {SF_DB}.{SF_SC}.firehose_nlp_labeled) 
#
CSR.execute(query)
total_rows = len(CSR.execute(query).fetch_pandas_all()) # execute once just to get total rows
print(f"{(total_rows):,} downloaded. Processing in batches...\n\n")

# again to start iterating over batches
CSR.execute(query)

c=0
current_pcnt = 0
total_rows_processed = 0
started_at = datetime.now()

print(f"Initiated Named-Entity Recognition (NER) analysis at\n{started_at}")
for batch in CSR.fetch_pandas_batches():
    c+=1
    total_rows_processed += len(batch)
    current_pcnt+=round((len(batch)/total_rows)*100, 1)
    print(f"\n{len(batch):,} rows downloaded from batch {(c):,} ({(current_pcnt):,.1f}% of total rows)")
    
    # using this thing as input is more efficient than using `batch` directly for whatever reason
    batch_dataset = Dataset.from_pandas(batch[['POST_TEXT']], preserve_index=False)
    # execute NER analysis 
    ner_output = PIPL_NER(batch_dataset['POST_TEXT'])
    # Add this col now to match schema-- will populate in the next step
    batch['SENTIMENT_ANALYSIS'] = None
    batch['NER_ANALYSIS']       = ner_output
    # match schema ordinal
    batch = batch[['CONTENT_ID','POST_CREATED_USA_TIMESTAMP','SENTIMENT_ANALYSIS','NER_ANALYSIS','POST_TEXT']]
    
    write_pandas(SF_XCT, batch
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
            ,use_logical_type = True
            ,auto_create_table = False
            ,overwrite = False
           )
    print(f"{len(batch):,} rows from batch written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")
    min_elapsed = (datetime.now() - started_at).total_seconds/60
    processing_velocity = round(total_rows_processed/min_elapsed, 2)
    print(f"Processing rate = {processing_velocity} rows/minute")

562,204 downloaded. Processing in batches...


Initiated Named-Entity Recognition (NER) analysis at
2025-06-05 17:04:23.526687

572 rows downloaded from batch 1 (0.1% of total rows)
572 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 0.4 rows/minute

1,677 rows downloaded from batch 2 (0.4% of total rows)
1,677 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 1.56 rows/minute

2,941 rows downloaded from batch 3 (0.9% of total rows)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


2,941 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 3.61 rows/minute

4,576 rows downloaded from batch 4 (1.7% of total rows)
4,576 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 6.8 rows/minute

11,584 rows downloaded from batch 5 (3.8% of total rows)
11,584 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 14.92 rows/minute

17,697 rows downloaded from batch 6 (6.9% of total rows)
17,697 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 27.45 rows/minute

34,856 rows downloaded from batch 7 (13.1% of total rows)
34,856 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 53.17 rows/minute

216 rows downloaded from batch 8 (13.1% of total rows)
216 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 53.34 rows/minute

517 rows downloaded from batch 9 (13.2% of total rows)
517 rows from batch written to bluesky_db.main

ForbiddenError: 000403: HTTP 403: Forbidden

## STEP 2

Apply Named-Entity Recognition (NER) and write to a stash table (`INT_FIREHOSE_NLP`)

In [ ]:
ner_output = PIPL_NER(DATA['POST_TEXT'].tolist())

# Add this col now to match schema-- will populate in the next step
DATA['SENTIMENT_ANALYSIS'] = None
DATA['NER_ANALYSIS']       = ner_output

write_pandas(SF_XCT, DATA
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")

KeyboardInterrupt: 

## STEP 3
Re-read data and apply Sentiment analysis, then writeback to stash

In [ ]:
query = f"select POST_TEXT,CONTENT_ID from {SF_DB}.{SF_SC}.int_firehose_nlp"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

sentiment_output = PIPL_SNT(DATA['POST_TEXT'].tolist())
DATA['SENTIMENT_ANALYSIS'] = sentiment_output

query=f"""
create temp table if not exists {SF_DB}.{SF_SC}.TMP_MERGE_SRC (
 POST_TEXT VARCHAR
,CONTENT_ID VARCHAR
,SENTIMENT_ANALYSIS VARIANT
)"""
CSR.execute(query)

write_pandas(SF_XCT, DATA
            ,table_name = 'TMP_MERGE_SRC'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.TMP_MERGE_SRC")

query=f"""
merge into {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP tgt
using {SF_DB}.{SF_SC}.TMP_MERGE_SRC src
   on src.content_id = tgt.content_id
when matched then update 
set tgt.SENTIMENT_ANALYSIS = src.SENTIMENT_ANALYSIS
"""
print(f"{(CSR.fetchone()[1]):,} rows updated in INT_FIREHOSE_NLP.SENTIMENT_ANALYSIS, using TMP_MERGE_SRC")

## STEP 4
 Retrieve stashed data, blow it out into many processed rows per-post, then insert.

In [ ]:
query = f"select * from {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

query = f"""
insert into {SF_DB}.{SF_SC}.firehose_nlp_labeled
with src as (
select a.content_id
      ,a.post_created_usa_timestamp
      ,b.readable_label_name as sentiment_detected_label
      ,cast(sentiment_analysis:score as number(5,4)) as sentiment_confidence_score
      ,row_number() over (
       partition by content_id
       order     by post_created_usa_timestamp, trim(a2.value:word, '"')
       ) as post_entity_number
      ,trim(a2.value:entity_group, '"') as ner_detected_group
      ,trim(a2.value:word, '"') as ner_detected_entity
      ,cast(a2.value:score as number(5,4)) as ner_confidence_score
from {SF_DB}.{SF_SC}.int_firehose_nlp a
left join table(flatten(input => parse_json(a.ner_analysis))) a2
left join {SF_DB}.{SF_SC}.label_map_roberta_base_sentiment b
       on trim(a.sentiment_analysis:label, '"') = b.model_label_name
)

select sha2(nvl(to_char(content_id), 'NULL') 
         || '||' 
         || nvl(to_char(post_entity_number), 'NULL')
       ) as analysis_id
      ,*
from src 
order by content_id
        ,post_created_usa_timestamp
        ,ner_detected_entity
"""
CSR.execute(query)

## Closing
Clear the stash, shut off the Snowflake Connection

In [ ]:
query = f'truncate table {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP'
CSR.execute(query)
SF_XCT.close()

# Unimplemented Code-Corral

In [ ]:
import dataset
from datetime import datetime, timedelta
import snowflake.connector
import transformers

def compute_ner_batches(ner_pipeline: transformers.pipelines.TokenClassificationPipeline) -> None:
    """
    Execute Named-Entity Recognition (NER) analysis on Bluesky Post data. Post-data is queried
    from Snowflake, then processed in batches. The pipeline is applied iteratively, to one batch
    of posts at a time. When each batch is completed, the new data is written back to 
    INT_FIREHOSE_NLP.

    This function leaves the SENTIMENT_ANALYSIS column 100% null; this is populated in a 
    subsequent function call.

    Args:
        ner_pipeline: This is a transfomer pipeline using Hugging Face model called 
                      "dslim/bert-base-NER"
    """
    
    # init some vars used to log process during runtime
    CSR.execute(f"""select count(*) from {SF_DB}.{SF_SC}.firehose_processed
                    where (first_detected_language = 'English'
                       or  first_detected_language is null
                       )
                      and post_created_usa_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                                         and to_timestamp_tz('2025-05-31 23:59:59+0000')     
                """)
    total_rows = CSR.fetchone()
    c = 0
    current_pcnt = 0
    rows_processed = 0
    
    # retrieve source data
    query = f"""
    select content_id
        ,usa_timestamp as post_created_usa_timestamp
        ,post_text
    from {SF_DB}.{SF_SC}.firehose_processed
    where (first_detected_language = 'English'
    or  first_detected_language is null
    )
    and post_created_usa_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                        and to_timestamp_tz('2025-05-31 23:59:59+0000')
    ;
    """
    CSR.execute(query)
    process_started_at = datetime.now()
    print(f"Initiated Named-Entity Recognition (NER) analysis at\n{process_started_at}")
    print(f"Downloading {(total_rows):,} total rows from FIREHOSE_PROCESSED...\n\n")
    
    # iterate through source data, doing batch-wise writebacks to the target table for each iteration
    for batch in CSR.fetch_pandas_batches():
        # more progress prints
        batch_started_at = datetime.now()
        c              += 1
        rows_processed += len(batch)
        current_pcnt   += round((len(batch)/total_rows)*100, 1)
        print(f"\n{len(batch):,} rows downloaded from batch {(c):,} ")
        
        # using this thing as input is more efficient than using `batch` directly for whatever reason
        batch_dataset = Dataset.from_pandas(batch[['POST_TEXT']], preserve_index=False)
        # execute NER analysis 
        ner_output = PIPL_NER(batch_dataset['POST_TEXT'])
        
        # capture data an execute writeback
        batch['SENTIMENT_ANALYSIS'] = None
        batch['NER_ANALYSIS']       = ner_output
        batch = batch[['CONTENT_ID','POST_CREATED_USA_TIMESTAMP','SENTIMENT_ANALYSIS','NER_ANALYSIS','POST_TEXT']]
        write_pandas(SF_XCT, batch
                ,table_name = 'INT_FIREHOSE_NLP'
                ,database   = SF_DB.replace('"', '').upper()
                ,schema     = SF_SC.replace('"', '').upper()
                ,use_logical_type = True
                ,auto_create_table = False
                ,overwrite = False
            )
        
        # more prints to show velocity 
        ## ... of this specific batch
        batch_finished_at = datetime.now()
        print(f"{len(batch):,} rows from batch written to INT_FIREHOSE_NLP ({(current_pcnt):,.1f}% of source rows processed)")
        print(f"Batch {(c):,} completed at {batch_finished_at}")
        second_time_for_batch = (batch_finished_at - batch_started_at).total_seconds
        minute_time_for_batch = second_time_for_batch // 60
        second_time_for_batch = second_time_for_batch % 60
        print(f"Batch Process Time = {minute_time_for_batch}min {second_time_for_batch}s")
       
        # ...of the entire process
        min_elapsed_since_process_start = (datetime.now() - process_started_at).total_seconds/60
        processing_velocity = round(rows_processed/min_elapsed, 2)
        print(f"Processing rate = {processing_velocity} rows/minute")
    
    total_seconds_for_process = (process_finished_at - process_started_at).total_seconds
    minute_time_for_process = total_seconds_for_process // 60
    second_time_for_process = total_seconds_for_process % 60
    print(f"{(c):,} batches totaling {(total_rows):,} rows were processed in {minute_time_for_process}min {second_time_for_process}s")
    print(f"Average process velocity is {round((total_rows/(total_seconds_for_process/60)), 1):,.1f}")

def compute_sentiment_batches_batches(sentiment_pipeline: transformers.pipelines.TokenClassificationPipeline) -> None:
    """
    Execute Sentiment (NER) analysis on Bluesky Post data. Post-data is queried from Snowflake, 
    then processed in batches. The pipeline is applied iteratively, to one batch of posts at a 
    time. When each batch is completed, the new data is merged into INT_FIREHOSE_NLP.

    To execute a MERGE from the Python Connector, this function must create a temp table, 
    insert into that, then merge the temp table to INT_FIREHOSE_NLP via a Cursor var.

    This function leaves the INT_FIREHOSE_NLP table fully populated. This table is 
    used to populate the final target table (FIREHOSE_NLP_LABELED) in a subsequent process.

    Args:
        sentiment_pipeline: This is a transfomer pipeline using Hugging Face model called 
                           "cardiffnlp/twitter-roberta-base-sentiment"
    """
    
    # init some vars used to log process during runtime
    CSR.execute(f"select count(*) from {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")
    total_rows = CSR.fetchone()
    c = 0
    current_pcnt = 0
    rows_processed = 0
    
    # create tmp table to use in later MERGE statement
    query = f"""create or replace temp table {SF_DB}.{SF_SC}.tmp_merge_src(
                POST_TEXT VARCHAR
               ,CONTENT_ID VARCHAR
               ,SENTIMENT_ANALYSIS VARIANT
               );
             """
    CSR.execute(query)
    
    # retrieve source data
    query = f"select content_id, post_text from {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP "
    CSR.execute(query)
    process_started_at = datetime.now()
    print(f"Initiated Sentiment analysis at\n{process_started_at}")
    print(f"Downloading {(total_rows):,} total rows from INT_FIREHOSE_NLP...\n\n")
    
    # iterate through source data, doing batch-wise writebacks to the target table for each iteration
    for batch in CSR.fetch_pandas_batches():
        # more progress prints
        batch_started_at = datetime.now()
        c              += 1
        rows_processed += len(batch)
        current_pcnt   += round((len(batch)/total_rows)*100, 1)
        print(f"\n{len(batch):,} rows downloaded from batch {(c):,} ")
        
        # using this thing as input is more efficient than using `batch` directly for whatever reason
        batch_dataset = Dataset.from_pandas(batch[['POST_TEXT']], preserve_index=False)
        # execute NER analysis 
        sent_output = PIPL_SENT(batch_dataset['POST_TEXT'])
        
        # capture data an execute writeback
        batch['SENTIMENT_ANALYSIS'] = sent_output
        batch = batch[['POST_TEXT','CONTENT_ID','SENTIMENT_ANALYSIS']]
        write_pandas(SF_XCT, batch
                ,table_name = 'TMP_MERGE_SRC'
                ,database   = SF_DB.replace('"', '').upper()
                ,schema     = SF_SC.replace('"', '').upper()
                ,use_logical_type = True
                ,auto_create_table = False
                ,overwrite = False
            )
        
        # more prints to show velocity 
        ## ... of this specific batch
        batch_finished_at = datetime.now()
        print(f"{len(batch):,} rows from batch written to INT_FIREHOSE_NLP ({(current_pcnt):,.1f}% of source rows processed)")
        print(f"Batch {(c):,} completed at {batch_finished_at}")
        second_time_for_batch = (batch_finished_at - batch_started_at).total_seconds
        minute_time_for_batch = second_time_for_batch // 60
        second_time_for_batch = second_time_for_batch % 60
        print(f"Batch Process Time = {minute_time_for_batch}min {second_time_for_batch}s")
       
        # ...of the entire process
        min_elapsed_since_process_start = (datetime.now() - started_at).total_seconds/60
        processing_velocity = round(rows_processed/min_elapsed, 2)
        print(f"Processing rate = {processing_velocity} rows/minute")
    
    process_finished_at = datetime.now()
    total_seconds_for_process = (process_finished_at - process_started_at).total_seconds
    minute_time_for_process = total_seconds_for_process // 60
    second_time_for_process = total_seconds_for_process % 60
    print(f"{(c):,} batches totaling {(total_rows):,} rows were processed in {minute_time_for_process}min {second_time_for_process}s")
    print(f"Average process velocity is {round((total_rows/(total_seconds_for_process/60)), 1):,.1f}")
            